# Week 05 - Decision Trees and Ensembles

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 06  
**Estimated time:** 3-4 hours

This notebook is self-contained. You will use a structured classification dataset to compare a single decision tree with bagging-style ensembles, then add the results to the numeric benchmark format.


## What You Are Building

This week has five required functions:

1. `split_data(X, y, test_size, random_state)` - stratified train/test split for classification.
2. `depth_accuracy_study(X_train, X_test, y_train, y_test, depths)` - measure how tree depth changes train/test performance.
3. `evaluate_classifiers(models, X_train, X_test, y_train, y_test)` - compare tree and ensemble models numerically.
4. `feature_importance_table(model, feature_names)` - extract and rank model feature importances.
5. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append classification results to the reusable benchmark format.

The modelling lesson is the bias-variance trade-off: a single deep tree can memorise, while ensembles often improve stability by averaging many trees.


In [ ]:
# Imports - keep this cell stable
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.25
DEPTHS = [1, 2, 3, 4, 5, 8, None]


## Dataset

Wine is a small structured classification dataset. The features are chemical measurements, and the target is wine cultivar.


In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target.rename("wine_class")
feature_names = list(X.columns)
class_names = list(wine.target_names)

print(X.shape)
display(X.head())
display(y.value_counts().sort_index().rename(index=dict(enumerate(class_names))))


## Task 1 - Stratified Split

Implement `split_data(X, y, test_size, random_state)`.

Return `(X_train, X_test, y_train, y_test)`. Use stratification so all classes are represented in both splits.


In [ ]:
def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Create a stratified train/test split for classification."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 1
X_train, X_test, y_train, y_test = split_data(X, y)

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert set(y_train.unique()) == set(y.unique())
assert set(y_test.unique()) == set(y.unique())
assert X_train.shape[1] == X.shape[1]

print("Task 1 passed")


## Task 2 - Tree Depth Study

Implement `depth_accuracy_study(X_train, X_test, y_train, y_test, depths)`.

For each depth, fit a `DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=depth)` and return a dataframe with columns:

- `max_depth`
- `train_accuracy`
- `test_accuracy`
- `train_f1_macro`
- `test_f1_macro`

Sort in the same order as `depths`.


In [ ]:
def depth_accuracy_study(X_train, X_test, y_train, y_test, depths: list) -> pd.DataFrame:
    """Evaluate decision-tree performance across max_depth values."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 2
depth_results = depth_accuracy_study(X_train, X_test, y_train, y_test, DEPTHS)

expected_cols = ["max_depth", "train_accuracy", "test_accuracy", "train_f1_macro", "test_f1_macro"]
assert isinstance(depth_results, pd.DataFrame)
assert list(depth_results.columns) == expected_cols
assert len(depth_results) == len(DEPTHS)
assert depth_results["train_accuracy"].between(0, 1).all()
assert depth_results["test_accuracy"].between(0, 1).all()
assert depth_results["train_accuracy"].max() >= depth_results["test_accuracy"].max()

print("Task 2 passed")
depth_results


In [ ]:
plot_depth = depth_results.copy()
plot_depth["depth_label"] = [
    "None" if pd.isna(depth) else str(int(depth))
    for depth in plot_depth["max_depth"]
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(plot_depth["depth_label"].tolist(), plot_depth["train_f1_macro"].to_numpy(), marker="o", label="train macro-F1")
ax.plot(plot_depth["depth_label"].tolist(), plot_depth["test_f1_macro"].to_numpy(), marker="o", label="test macro-F1")
ax.set_xlabel("max_depth")
ax.set_ylabel("macro-F1")
ax.set_title("Decision tree depth study")
ax.legend()
plt.tight_layout()
plt.show()


## Guided Analysis: Inspect One Tree

Fit the best-depth tree according to test macro-F1 and inspect its first few rules.


In [ ]:
best_depth = depth_results.sort_values("test_f1_macro", ascending=False).iloc[0]["max_depth"]
best_depth_param = None if pd.isna(best_depth) else int(best_depth)

tree_model = DecisionTreeClassifier(max_depth=best_depth_param, random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

print("Best depth:", best_depth_param)
print(export_text(tree_model, feature_names=feature_names).splitlines()[:20])

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
)
plt.tight_layout()
plt.show()


## Task 3 - Evaluate Tree Ensembles

Implement `evaluate_classifiers(models, X_train, X_test, y_train, y_test)`.

Output columns:

- `model`
- `accuracy`
- `f1_macro`
- `fit_time_sec`

Fit each model on the training set and evaluate on the test set. Sort by `f1_macro` descending.


In [ ]:
def evaluate_classifiers(models: dict, X_train, X_test, y_train, y_test) -> pd.DataFrame:
    """Fit classifiers and return comparable held-out test metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 3
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    f"decision_tree_depth_{best_depth_param}": DecisionTreeClassifier(max_depth=best_depth_param, random_state=RANDOM_STATE),
    "decision_tree_unpruned": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "bagging_trees_100": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=100,
        random_state=RANDOM_STATE,
    ),
    "random_forest_200": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "extra_trees_200": ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

model_results = evaluate_classifiers(models, X_train, X_test, y_train, y_test)

assert isinstance(model_results, pd.DataFrame)
assert list(model_results.columns) == ["model", "accuracy", "f1_macro", "fit_time_sec"]
assert len(model_results) == len(models)
assert model_results["f1_macro"].is_monotonic_decreasing
assert model_results[["accuracy", "f1_macro", "fit_time_sec"]].notna().all().all()
assert model_results.loc[model_results["model"] == "random_forest_200", "accuracy"].iloc[0] > 0.85

print("Task 3 passed")
model_results


## Task 4 - Feature Importance Table

Implement `feature_importance_table(model, feature_names)`.

Fit models before calling this function. Return a dataframe with columns:

- `feature`
- `importance`
- `rank`

Sort by importance descending. Rank starts at 1.


In [ ]:
def feature_importance_table(model, feature_names: list[str]) -> pd.DataFrame:
    """Return sorted feature importances for a fitted tree-based model."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 4
rf_model = models["random_forest_200"]
rf_model.fit(X_train, y_train)
importance_df = feature_importance_table(rf_model, feature_names)

assert isinstance(importance_df, pd.DataFrame)
assert list(importance_df.columns) == ["feature", "importance", "rank"]
assert len(importance_df) == len(feature_names)
assert importance_df["importance"].is_monotonic_decreasing
assert importance_df["rank"].tolist() == list(range(1, len(feature_names) + 1))
assert np.isclose(importance_df["importance"].sum(), 1.0)

print("Task 4 passed")
importance_df.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=importance_df.head(10), x="importance", y="feature", ax=ax, color="steelblue")
ax.set_title("Random forest feature importances")
plt.tight_layout()
plt.show()


## Task 5 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

Convert model results into long benchmark format:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

Include numeric metric columns such as `accuracy` and `f1_macro`. Do not include `fit_time_sec` as a metric.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert classifier results to the cumulative benchmark long format."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 5
benchmark_long = make_benchmark_long(
    model_results,
    week="W05",
    dataset="Wine",
    task_type="classification",
    target="wine_class",
    split="75_25_stratified_random_state_42",
)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert {"accuracy", "f1_macro"}.issubset(set(benchmark_long["metric"]))
assert "fit_time_sec" not in set(benchmark_long["metric"])
assert benchmark_long["score"].between(0, 1).all()

print("Task 5 passed")
benchmark_long.head(10)


## Benchmark Wide View

Each tree or ensemble model becomes a column. This W05 table can be concatenated with compatible W03 Wine classification results if the split and metric choices are aligned.


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Guided Analysis: Confusion Matrix

Inspect the best tree-based model's mistakes.


In [ ]:
best_model_name = model_results.iloc[0]["model"]
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## Reflection

Answer briefly, but concretely.

1. What changed as tree depth increased? Where do you see overfitting?
2. Did the ensemble models improve over a single tree on Wine?
3. Which features did the random forest rely on most, and does that match the inspected tree?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Out-of-Bag Score
Train a random forest with `oob_score=True`. Compare its OOB score with the test-set score.

### Track B - Another Dataset
Apply the same tree benchmark to the Breast Cancer dataset from sklearn.

### Track C - Preview Boosting
Add `GradientBoostingClassifier` to the benchmark as a preview of Week 06.


In [ ]:
# Optional challenge workspace
# Your code here
